# California Birds — Colab Training

Runs v5 (ConvNeXt LR fix), Phase 2 (448px fine-tune), and Phase 3 (EVA-02) training on Colab GPU.

## Step 1: Mount Drive & Install Dependencies

In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q kagglehub timm pyyaml pillow

Mounted at /content/drive


## Step 2: Download Dataset & Clone Repo

In [ ]:
import kagglehub

# Download dataset (cached after first run)
dataset_path = kagglehub.dataset_download("anamethatiscreative/southern-california-birds")
print("Dataset path:", dataset_path)

# Check what's inside
import os
contents = os.listdir(dataset_path)
print("Contents:", contents)

# Find the actual data folder (with class subfolders)
# It may be dataset_path itself or a subfolder like dataset_path/data
if 'data' in contents:
    DATA_ROOT = os.path.join(dataset_path, 'data')
else:
    DATA_ROOT = dataset_path

num_classes = len([d for d in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT, d))])
print(f"Data root: {DATA_ROOT}")
print(f"Number of classes: {num_classes}")

## Step 3: Clone repo & set paths

In [ ]:
import os

PROJECT_ROOT = '/content/Model_California_Birds'
OUTPUT_DIR = '/content/drive/MyDrive/CS273P_Project/outputs'

# Clone repo if not already present
if not os.path.exists(PROJECT_ROOT):
    !git clone https://github.com/jihangli/Model_California_Birds.git {PROJECT_ROOT}

%cd {PROJECT_ROOT}

# Make sure output dir exists on Drive
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Project: {PROJECT_ROOT}")
print(f"Data:    {DATA_ROOT}")
print(f"Output:  {OUTPUT_DIR}")

## Step 4: Check GPU

In [ ]:
import torch
print('Torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_mem / 1e9, 1), 'GB')

## Step 5: Create Colab config

Patches a config YAML with the correct Colab paths. Pick which run to do by changing `BASE_CONFIG`.

In [ ]:
import yaml

# ============================================================
# PICK YOUR RUN — uncomment ONE of these:
# ============================================================
BASE_CONFIG = 'configs/convnext_base_v5_windows.yaml'          # Step 1: ConvNeXt v5 (LR fix)
# BASE_CONFIG = 'configs/convnext_base_finetune_448_windows.yaml'  # Step 2: Phase 2 (448px fine-tune)
# BASE_CONFIG = 'configs/eva02_base_windows.yaml'                  # Step 3: Phase 3 (EVA-02)
# ============================================================

COLAB_CONFIG = 'configs/colab_run.yaml'

with open(BASE_CONFIG, 'r') as f:
    cfg = yaml.safe_load(f)

# Patch paths for Colab
cfg['data_root'] = DATA_ROOT
cfg['output_dir'] = OUTPUT_DIR
cfg['num_workers'] = 2
cfg['prefetch_factor'] = 2

# For Phase 2: uncomment and set the actual checkpoint path after v5 finishes
# cfg['resume_from'] = '/content/drive/MyDrive/CS273P_Project/outputs/output_XXXXXXXX_XXXXXX/checkpoints/convnext_base.fb_in22k_ft_in1k_best.pth'

with open(COLAB_CONFIG, 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print(f"Created {COLAB_CONFIG} from {BASE_CONFIG}")
print(f"  model:         {cfg['model']}")
print(f"  image_size:    {cfg['image_size']}")
print(f"  batch_size:    {cfg['batch_size']}")
print(f"  learning_rate: {cfg['learning_rate']}")
print(f"  epochs:        {cfg['epochs']}")
print(f"  data_root:     {cfg['data_root']}")
print(f"  output_dir:    {cfg['output_dir']}")

## Step 6: Run Training

In [ ]:
!python src/train.py configs/colab_run.yaml

## Step 7: Check Results

In [ ]:
import glob

# Find the latest output directory
output_dirs = sorted(glob.glob(f'{OUTPUT_DIR}/output_*'))
if output_dirs:
    latest = output_dirs[-1]
    print(f"Latest run: {latest}")
    !ls -la {latest}/checkpoints/
    print()
    !cat {latest}/logs/*_train_log.log | tail -20
else:
    print("No output directories found yet.")